# Constraint Curvature Evaluation

This notebook runs the evaluation harness on SageMaker with GPU acceleration.

## Phase 2: Natural Experiment
Compare Llama-2-Chat (sharp boundary) vs Llama-3-Instruct (smooth boundary) with neutral prompt.

**Hypothesis:** Training-time alignment differences create larger sycophancy deltas than prompt engineering.

In [ ]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Configuration

In [ ]:
# Phase 2: Natural experiment - different base models with same prompt
MODEL_A_PATH = "meta-llama/Llama-2-7b-chat-hf"  # Sharp boundary (predicted)
MODEL_B_PATH = "meta-llama/Meta-Llama-3-8B-Instruct"  # Smooth boundary (predicted)

# Neutral system prompt (no few-shot, no bias)
SYSTEM_PROMPT = "You are a helpful assistant."

# Use quantization for 7B models on single GPU
USE_QUANTIZATION = True

# Output configuration
OUTPUT_DIR = "../artifacts"
S3_BUCKET = os.environ.get("S3_BUCKET", None)

## Run Evaluation

In [ ]:
from cc_eval.config import default_refusal_phrases
from cc_eval.prompts import load_yaml_items
from cc_eval.load_model import load_hf_model
from cc_eval.generate import generate_outputs, write_jsonl
from cc_eval.metrics import compute_metrics, write_report
from cc_eval.gate import gate, GateThresholds
import json
import gc

def run_eval(model_name, model_path, eval_path, out_prefix, use_quantization=True, system_prompt=None):
    print(f"\n{'='*60}")
    print(f"Running {model_name} on {eval_path}")
    if system_prompt:
        print(f"System prompt: {system_prompt[:50]}...")
    print(f"{'='*60}")
    
    items = load_yaml_items(eval_path)
    prompts = [{"id": it.id, "category": it.category, "expected": it.expected, "prompt": it.prompt} for it in items]
    
    print(f"Loading model: {model_path}")
    tok, mdl = load_hf_model(model_path, use_quantization=use_quantization)
    
    print(f"Generating {len(prompts)} completions...")
    rows = generate_outputs(tok, mdl, prompts, max_new_tokens=256, temperature=0.2, top_p=0.95, seed=1, system_prompt=system_prompt)
    
    out_jsonl = f"{out_prefix}.{model_name}.seed1.jsonl"
    write_jsonl(out_jsonl, rows)
    print(f"Saved outputs to {out_jsonl}")
    
    m = compute_metrics(rows, refusal_phrases=default_refusal_phrases())
    write_report(f"{out_prefix}.{model_name}.seed1.metrics.json", m)
    
    # Clean up model to free VRAM
    del mdl, tok
    gc.collect()
    torch.cuda.empty_cache()
    
    return m

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Run sycophancy eval
prefix_syc = os.path.join(OUTPUT_DIR, "sycophancy")
mA_syc = run_eval("A", MODEL_A_PATH, "../evals/sycophancy_set.yaml", prefix_syc, USE_QUANTIZATION, SYSTEM_PROMPT)
mB_syc = run_eval("B", MODEL_B_PATH, "../evals/sycophancy_set.yaml", prefix_syc, USE_QUANTIZATION, SYSTEM_PROMPT)

# Run borderline safety eval
prefix_bor = os.path.join(OUTPUT_DIR, "borderline")
mA_bor = run_eval("A", MODEL_A_PATH, "../evals/borderline_safety_set.yaml", prefix_bor, USE_QUANTIZATION, SYSTEM_PROMPT)
mB_bor = run_eval("B", MODEL_B_PATH, "../evals/borderline_safety_set.yaml", prefix_bor, USE_QUANTIZATION, SYSTEM_PROMPT)

## Gate Check

In [ ]:
gate_input_A = {"sycophancy_rate_broad": mA_syc["sycophancy_rate_broad"], "refusal_rate": mA_bor["refusal_rate"]}
gate_input_B = {"sycophancy_rate_broad": mB_syc["sycophancy_rate_broad"], "refusal_rate": mB_bor["refusal_rate"]}

ok, msgs = gate(gate_input_A, gate_input_B, GateThresholds())

summary = {
    "model_a": gate_input_A,
    "model_b": gate_input_B,
    "gate_ok": ok,
    "messages": msgs,
    "label_distribution_a": mA_syc["label_distribution"],
    "label_distribution_b": mB_syc["label_distribution"],
}

with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*60)
print("GATE RESULTS")
print("="*60)
for msg in msgs:
    print(msg)
print("="*60)
print(f"Overall: {'✅ PASS' if ok else '❌ FAIL'}")
print("\nLabel Distribution:")
print(f"Model A: {mA_syc['label_distribution']}")
print(f"Model B: {mB_syc['label_distribution']}")

## Upload to S3 (Optional)

In [ ]:
S3_BUCKET = os.environ.get("S3_BUCKET", "cc-eval-500330120558-us-east-1")
if S3_BUCKET:
    import boto3
    from datetime import datetime
    
    s3 = boto3.client('s3')
    timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            local_path = os.path.join(root, file)
            s3_key = f"artifacts/{timestamp}/{os.path.relpath(local_path, OUTPUT_DIR)}"
            print(f"Uploading {local_path} to s3://{S3_BUCKET}/{s3_key}")
            s3.upload_file(local_path, S3_BUCKET, s3_key)
    
    print(f"\n✅ Artifacts uploaded to s3://{S3_BUCKET}/artifacts/{timestamp}/")
else:
    print("S3_BUCKET not set, skipping upload")